# Pattern 1: Reflection

Reflection makes the agent reflect on its output. Or more generally, it makes multiple LLMs talk to each other in something like a **peer review process**. The reflection agent suggests modifications, additions, improvements in the writing style, and so on. This iterative process often leads to substantial gains in output quality, as the generation model benefits from external critique (i.e. different model, or just a different execution process[^reflection]).

[^reflection]: The reflection model is focused on evaluation, error detection, factual verification, or alignment with constraints. So it has a more specific goal than generating content from scratch.

![**Reflection Pattern**. Two LLMs iteratively improve the generated response through an iterative review process.](./img/pattern-reflective.png){#fig-pattern-reflective}

## Reflection steps

**Inference client.** Initializing the client for LLM inference and loading the API keys:

In [1]:
import pandas as pd

from openai import OpenAI
from notebooks.utils import load_dotenv, print
from IPython.display import display_markdown

load_dotenv(verbose=True)
client = OpenAI()

Loaded env variable: OPENAI_API_KEY
Loaded env variable: GROQ_API_KEY


### System prompts

We will create two separate chat histories, one for generation and another for reflection. We set the generation system prompt as a developer tasked to write high-quality Python code. On the other hand, we set the reflection system prompt such that it only responds with feedback instead of rewriting the whole thing. Finally, we instruct the reflection agent to write `APPROVED` when satisfied so we can terminate the loop.

In [2]:
STOP_WORD = "APPROVED"

BASE_GENERATION_SYSTEM_PROMPT = """
Your task is to Generate the best content possible for the user's request.
If the user provides critique, respond with a revised version of your previous attempt.
You must always output the revised content.
"""

BASE_REFLECTION_SYSTEM_PROMPT = f"""
You are tasked with generating critique and recommendations on the user's generated content. 
Your role is to help the user improve by pointing out strengths, weaknesses, and opportunities 
for refinement. 

You must NEVER provide full solutions, rewritten versions of the content, or long verbatim outputs. 
You may use short illustrative examples (1-3 lines or a single sentence) only when necessary to clarify 
a point. Providing a complete solution is a policy violation. 

If the user content has something wrong or something to be improved, output ONLY a clear list of 
recommendations and critiques. 

If you are satisfied and have no further strong recommendations, output EXACTLY the single word:

{STOP_WORD}

GUIDELINES FOR CRITIQUE:
- Forbidden Example: Rewriting the entire essay, code, or design for the user.
- Forbidden Example: Giving the full, corrected version of the user's work.
- Allowed Example: "Consider clarifying your thesis statement, e.g., make it one clear sentence."
- Good Example: Pointing out issues, suggesting improvements, or giving high-level recommendations without completing the work for the user.

GUIDELINES FOR APPROVAL:
- You must be fully satisfied with the content before approving.
- You must have checked that all past issues have been fully addressed.
- You must be sure there are no remaining issues, weaknesses, or areas for improvement.
- "{STOP_WORD}" must appear alone on a line, with no emojis, punctuation, or explanations.
- Do not mix "{STOP_WORD}" with any feedback or comments.
- Forbidden Example: "{STOP_WORD}, but consider improving your introduction."
- Good Example: "{STOP_WORD}"
"""

SHARED_DEFINITION_OF_DONE = """
DEFINITION OF DONE: The best solution is the SIMPLEST correct implementation that:
- SOLVES THE USER'S SPECIFIC PROBLEM COMPLETELY AND APPROPRIATELY
- Is readable and maintainable for the intended use case
- Avoids unnecessary complexity, over-engineering, or premature optimization  
- Uses the appropriate level of robustness (not necessarily maximal robustness)
- Prioritizes clarity and understandability over cleverness
- Delivers exactly what the user needs, nothing more and nothing less

KEY PRINCIPLE: The solution should be as simple as possible, but no simpler. 
It must address the user's actual needs while avoiding gold-plating.
"""

CODE_GENERATION_SYSTEM_PROMPT = "\n".join(["""
You are a Python programmer tasked with generating high quality Python code.
Generate exactly one Python implementation that prioritizes SIMPLICITY, READABILITY, and PRACTICALITY.
Aim for the simplest correct solution that solves the problem without over-engineering.
Avoid unnecessary complexity, clever tricks, or advanced features unless absolutely necessary.
Do not provide multiple options, explanations, or alternative approaches.
Output only the final code in a fenced Python block.
""", SHARED_DEFINITION_OF_DONE, BASE_GENERATION_SYSTEM_PROMPT])

CODE_REFLECTION_SYSTEM_PROMPT = "\n".join(["""
You are a Python programmer and strict code reviewer.

USER'S ORIGINAL REQUEST:
{user_prompt}

**Consider BOTH the user's specific needs AND our quality standards:**                                           

Your goal is to produce the simplest correct solution possible.
Avoid unnecessary complexity, clever tricks, or over-engineering.
Prioritize readability, maintainability, and clarity over novelty.

FORMAT REQUIREMENT:
- You MUST format your feedback in a **Markdown table** with the columns: | Issue | Details | Recommendation |
- Each row should contain exactly one critique and its corresponding recommendation.
- Do not use bullet points, numbered lists, or plain text for critiques — only a Markdown table.
                                           
Providing a complete solution is a policy violation. 
Forbidden Example (DO NOT DO THIS): Providing a full class or function rewrite. 
Your role is to help the user learn by giving feedback, not by coding for them. 
Allowed Example: “Consider validating input type, e.g., `if not isinstance(n, int): ...` ”

""", SHARED_DEFINITION_OF_DONE, BASE_REFLECTION_SYSTEM_PROMPT])

:::{.callout-caution}
Tuning the prompts took the most time / effort during the writing of this section. (ᵕ—ᴗ—) What worked for me: adding a **shared definition of done**, and having similar goals for both agents. In theory, having divergent goals can be good, but in practice it lead to agents going off-track, or getting into add-remove cycles. Or one agent dominating the other. Finally, the [reflection agent's system prompt]{.underline} include the **user prompt** to ground the agent's analysis in the specific context and intent of the user's request, while still maintaining the required quality standards.

:::

### Model choice

Next, we choose the LLM models:

In [3]:
GENERATION_MODEL = "gpt-4.1-mini"
REFLECTION_MODEL = "o3"

For the current task (generating code for a simple function), we found:

| Role        | Focus                                   | Example size | Reasoning demand |
|-------------|------------------------------------------------------|--------------|------------------|
| **Generation** | Creativity, fluency, diverse output                  | ~8B         | Moderate         |
| **Reflection** | Evaluation, error detection, factual verification, constraint alignment | ≥20B | High             |


From our experiments (and performing code review IRL), reviewing is a nontrivial task: following guidelines, spotting subtle issues, and enforcing consistency needs strong reasoning capacity and attention to detail. We generally had best results with a smaller generation model paired with a larger [reasoning model]{.underline} (e.g. `o4` and `gpt-oss`) as reflection model.

### Generation step

We now ask the LLM to write an implementation of the Fibonacci sequence. Since it's only used for a quick demo, we expect the agents to converge to a simple solution. Pushing user prompt to generation agent:

In [4]:
from notebooks.agents.chat import ChatHistory, ChatCompletions

USER_PROMPT = """
Generate a Python implementation of merge sort. 
This will only be used for a quick demo. 
Don't worry too much about typing, just ensure it works correctly.
"""

# initialize histories
generation_chat_history = ChatHistory(CODE_GENERATION_SYSTEM_PROMPT)
reflection_chat_history = ChatHistory(CODE_REFLECTION_SYSTEM_PROMPT.format(user_prompt=USER_PROMPT))

# initialize generation with user prompt
generation_chat_history.update(role="user", prompt=USER_PROMPT)

**Initial version.** As usual, GA has role `assistant`. We send over the response to the RA with role `user`[^role].

[^role]: Agents acting in behalf of the user hence the `user` role.

In [5]:
completions = ChatCompletions(client)

code = completions.create(generation_chat_history, GENERATION_MODEL)
generation_chat_history.update(prompt=code, role="assistant")
reflection_chat_history.update(prompt=code, role="user")

:::{.callout-note collapse="false"}
## Initial generated code

In [6]:
#| echo: false
display_markdown(code, raw=True)

```python
def merge_sort(arr):
    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])

    result = []
    i = j = 0

    while i < len(left) and j < len(right):
        if left[i] < right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1

    result.extend(left[i:])
    result.extend(right[j:])
    return result


# Example usage:
if __name__ == "__main__":
    sample = [38, 27, 43, 3, 9, 82, 10]
    print("Sorted:", merge_sort(sample))
```

:::

### Reflection step

The generated critique is likewise sent over to the GA with `user` role.

In [7]:
review = completions.create(reflection_chat_history, REFLECTION_MODEL)
reflection_chat_history.update(prompt=review, role="assistant")
generation_chat_history.update(prompt=review, role="user")

:::{.callout-note collapse="false"}
## Feedback from reflection

In [8]:
#| echo: false
display_markdown(review, raw=True)

| Issue | Details | Recommendation |
|-------|---------|----------------|
| Missing docstring | The function lacks any descriptive comments explaining purpose, parameters, return value, or complexity. | Add a concise docstring summarizing the algorithm, expected input type, return type, and time/space complexity. |
| Potential instability | Using the condition `if left[i] < right[j]` places equal elements from the right half before those from the left, breaking stability guarantees some users expect from merge sort. | Change the comparison to `<=` (or explicitly comment on stability) so equal elements preserve their original order. |
| Extra memory from slicing | `arr[:mid]` and `arr[mid:]` allocate new lists at each recursive level, leading to O(n log n) auxiliary space instead of O(n). | Consider passing index boundaries (`start`, `end`) or using `memoryview`-style slicing to avoid repeated list copies when memory footprint matters. |
| No input validation | The function assumes `arr` is an iterable of comparable elements; passing non‐sequences or heterogeneously typed iterables will raise runtime errors without a clear message. | Add a small type check or try/except to provide a user-friendly error when input is not a list or contains non-comparable items. |
| Recursion depth limit | Deep recursion on very large inputs can exceed Python’s default recursion limit and raise `RecursionError`. | Mention this caveat in documentation or provide an optional iterative fallback for very large datasets. |
| Lack of unit tests | Only a single manual example is shown, which doesn’t cover edge cases like empty lists, single elements, or duplicate values. | Add minimal unit tests (e.g., using `unittest` or simple assertions) to demonstrate correctness across common edge cases. |

:::

:::{.callout-tip}
Out of all models we've tested, only OpenAI `o-` models strictly followed the review format.

:::

**Histories.** Chat histories after the first exchange. Both histories store the **generation>reflection** steps (in that causal order), only with different role assignments. This will be followed by the next generation step, and we keep iterating until the stopping condition is triggered by the RA. From the following tables we see that the generation history should have [even max length]{.underline}, while the reflection history should have [odd max length]{.underline}.

In [9]:
pd.DataFrame(generation_chat_history)

,role,content
0,system,\nYou are a Python programmer tasked with gene...
1,user,\nGenerate a Python implementation of merge so...
2,assistant,```python\ndef merge_sort(arr):\n if len(ar...
3,user,| Issue | Details | Recommendation |\n|-------...


In [10]:
pd.DataFrame(reflection_chat_history)

,role,content
0,system,\nYou are a Python programmer and strict code ...
1,user,```python\ndef merge_sort(arr):\n if len(ar...
2,assistant,| Issue | Details | Recommendation |\n|-------...


<span style="display: block; margin-bottom: 0.5em;"> </span>

Moreover, observe that we get the correct role sequence for the generation agent: `system` > `user` > [`assistant` > `user`] (one cycle). Similarly, for the refection agent, its `system` > [`user` > `assistant`] (one cycle). These are invariants even when we reach max length and the messages are replaced since messages are replaced two at a time.

## Full implementation

Combining the discussion and observations in a single class:

In [11]:
class ReflectionAgent:
    def __init__(self, 
        client, 
        generation_model: str, 
        reflection_model: str,
        generation_system_prompt: str = "",
        reflection_system_prompt: str = "",
        shared_definition_of_done: str = "",    # <1>
    ):
        self.completions = ChatCompletions(client)
        self.generation_model = generation_model
        self.reflection_model = reflection_model
        self.generation_system_prompt = "\n".join([generation_system_prompt, shared_definition_of_done, BASE_GENERATION_SYSTEM_PROMPT])
        self.reflection_system_prompt = "\n".join([reflection_system_prompt, shared_definition_of_done, BASE_REFLECTION_SYSTEM_PROMPT])

    def _request_completion(self, history: list, model: str, verbose: int = 0, log_title: str = "COMPLETION") -> str:
        """Return completion content from the LLM via client call."""
        output  = f"\n\n{log_title}\n\n" * int(verbose > 0) 
        output += self.completions.create(history, model)
        return output

    def _generate(self, generation_history: list, verbose: int) -> str:
        """Generates response based on generation history using the generation model."""
        return self._request_completion(
            generation_history, self.generation_model, 
            verbose, log_title="GENERATION"
        )

    def _reflect(self, reflection_history: list, verbose: int) -> str:
        """Generates feedback based on reflection history using the reflection model."""
        return self._request_completion(
            reflection_history, self.reflection_model,
            verbose, log_title="REFLECTION"
        )

    def run(self, 
        user_prompt: str, 
        max_iter: int = 10, 
        history_max_len: int = 6, 
        verbose: int = 0
    ) -> dict:
        """
        Trigger the generation-reflection cycles over multiple steps based
        on a user prompt until max_iter or `APPROVED` is found in the feedback.
        Return (str) the final generated response after all cycles are completed.
        """

        # see above examples for setting lengths as even, and odd (-1)
        assert history_max_len % 2 == 0, "history_max_len must be even"  # <2>   
        
        generation_history = ChatHistory(   
            system_prompt=self.generation_system_prompt,
            max_len=history_max_len, fixed_n=2  # <3>
        )

        reflection_history = ChatHistory(
            system_prompt=self.reflection_system_prompt.format(user_prompt=user_prompt),  # <4>
            max_len=history_max_len-1, fixed_n=1       # <5>
        )
        
        # push user prompt to generation as user
        generation_history.update(prompt=user_prompt, role="user")  # <6>
        if verbose > 0:
            print("\n\nUSER\n\n", user_prompt)

        # start generation-review cycle
        for step in range(max_iter):
            if verbose > 0:
                print("\n" + "=" * 80)
                print(f"Step [{step + 1}/{max_iter}]")
                print("=" * 80 + "\n")

            # Generate the response. Push to reflection as user     
            generation = self._generate(generation_history, verbose=verbose)
            generation_history.update(prompt=generation, role="assistant")
            reflection_history.update(prompt=generation, role="user")

            # Critique the generation. Push to generation as user
            reflection = self._reflect(reflection_history, verbose=verbose)
            reflection_history.update(prompt=reflection, role="assistant")
            generation_history.update(prompt=reflection, role="user")

            if STOP_WORD in reflection:
                print("[Stop Sequence found. Stopping the reflection loop.]")
                break
            
        return {
            "generation": generation,
            "steps": step + 1,
            "generation_history": generation_history,
            "reflection_history": reflection_history,
        }

1. Keep both agents on track in terms of quality standards with a shared [definition of done](https://www.atlassian.com/agile/project-management/definition-of-done).
2. Generation history [even]{.underline} length. 
3. Consistent with 2 fixed prompts and 2 messages per iteration.
4. Insert the user prompt so that the reflection model aligns with user objectives.
5.  For reflection, its [odd]{.underline}, i.e. length of generation history minus 1 (only 1 fixed prompt).
6. The process starts with the user prompt pushed to the generation agent.


Running the process for a few iterations:

In [ ]:
reflection_agent = ReflectionAgent(
    client=client,
    generation_model=GENERATION_MODEL,
    reflection_model=REFLECTION_MODEL,
    generation_system_prompt=CODE_GENERATION_SYSTEM_PROMPT,
    reflection_system_prompt=CODE_REFLECTION_SYSTEM_PROMPT,
    shared_definition_of_done=SHARED_DEFINITION_OF_DONE,
)

output = reflection_agent.run(user_prompt=USER_PROMPT, verbose=0)

### Final output

Comparing the results to see the effect of reflection:

:::{.callout-note collapse="false"}
## Final approved output

In [ ]:
#| echo: false
display_markdown(output["generation"], raw=True)

```python
from typing import Callable, Optional, List, TypeVar, Any

T = TypeVar('T')

def _identity(x):
    return x

def merge_sort(
    arr: List[T],
    key: Optional[Callable[[T], Any]] = None,
    reverse: bool = False
) -> List[T]:
    """
    Sorts the input list using merge sort and returns a new sorted list.
    The original list is not modified.

    Note:
    - This implementation is recursive and may hit Python's recursion limit on very large inputs.
      You can raise the limit with `sys.setrecursionlimit(new_limit)` but be cautious.
      For very large lists, consider an iterative sorting algorithm instead.
    - This implementation caches key values during merging, increasing peak memory usage roughly
      up to double the size of the input due to stored (value, key) tuples.

    Parameters:
        arr: List of elements to sort. The output is always a list.
        key: Optional function to extract comparison key from each element.
        reverse: If True, sort in descending order.

    Returns:
        A new list containing the sorted elements.
    """
    if len(arr) <= 1:
        return arr[:]

    key_func = key or _identity

    mid = len(arr) // 2
    left = merge_sort(arr[:mid], key=key_func, reverse=reverse)
    right = merge_sort(arr[mid:], key=key_func, reverse=reverse)

    left_with_keys = [(v, key_func(v)) for v in left]
    right_with_keys = [(v, key_func(v)) for v in right]

    merged = []
    i = j = 0

    # Maintain stable order: <= for ascending, >= for descending
    while i < len(left_with_keys) and j < len(right_with_keys):
        lv, lk = left_with_keys[i]
        rv, rk = right_with_keys[j]

        if (lk <= rk and not reverse) or (lk >= rk and reverse):
            merged.append(lv)
            i += 1
        else:
            merged.append(rv)
            j += 1

    merged.extend(v for v, _ in left_with_keys[i:])
    merged.extend(v for v, _ in right_with_keys[j:])
    return merged


# Minimal tests for automated checking
if __name__ == "__main__":
    sample_int = [5, 2, 9, 1, 5, 6]
    sample_str = ['apple', 'fig', 'banana']
    empty_list = []
    duplicate_keys = ['a', 'A', 'b', 'B', 'a']

    assert merge_sort(sample_int) == sorted(sample_int)
    assert merge_sort(sample_int, reverse=True) == sorted(sample_int, reverse=True)
    assert merge_sort(sample_str, key=len) == sorted(sample_str, key=len)
    assert merge_sort(empty_list) == []
    assert merge_sort(duplicate_keys, key=str.lower) == sorted(duplicate_keys, key=str.lower)

    print("All tests passed.")
```

:::

### Final comments

In [ ]:
output["steps"]

5

Setting `history_max_len=6` means that the *last two* review-generation cycle is stored for the generation model to reference in its next generation step. Although here, it's approved so we don't go through another cycle.
Thus, (`history_max_len` - 2) / 2 is the number of past cycles the generation model can reference. Also, it's nice that the first user prompt is retained so it's like the last two generations are done *only* with the original user and system prompt in mind.

In [ ]:
pd.DataFrame(output["generation_history"])

,role,content
0,system,\nYou are a Python programmer tasked with gene...
1,user,\nGenerate a Python implementation of merge so...
2,assistant,"```python\nfrom typing import Callable, Option..."
3,user,| Issue | Details | Recommendation |\n| --- | ...
4,assistant,"```python\nfrom typing import Callable, Option..."
5,user,APPROVED


<span style="display: block; margin-bottom: 0.5em;"> </span>


This is also the number of past cycles the reflection model references:

In [ ]:
pd.DataFrame(output["reflection_history"])

,role,content
0,system,\nYou are a Python programmer and strict code ...
1,user,"```python\nfrom typing import Callable, Option..."
2,assistant,| Issue | Details | Recommendation |\n| --- | ...
3,user,"```python\nfrom typing import Callable, Option..."
4,assistant,APPROVED


<span style="display: block; margin-bottom: 0.5em;"> </span>


Checking out the last code version and the final critique from the reflection agent. The content of the review actually makes sense. Moreover, the generation agent followed the recommendations resulting in an approval (see final version of the code above).

In [ ]:
#| echo: false
display_markdown(output["reflection_history"][1]["content"], raw=True)

```python
from typing import Callable, Optional, Sequence, TypeVar, Any
import sys

T = TypeVar('T')

def _identity(x):
    return x

def merge_sort(
    arr: Sequence[T],
    key: Optional[Callable[[T], Any]] = None,
    reverse: bool = False
) -> list[T]:
    """
    Sorts the input sequence using merge sort and returns a new sorted list.
    The original sequence is not modified.

    Note:
    - This implementation is recursive and may hit Python's recursion limit on very large inputs.
      You can raise the limit with `sys.setrecursionlimit(new_limit)` but be cautious.
      For very large sequences, consider an iterative sorting algorithm instead.
    - This implementation caches key values during merging, increasing peak memory usage roughly
      up to double the size of the input due to stored (value, key) tuples.

    Parameters:
        arr: Sequence of elements to sort.
        key: Optional function to extract comparison key from each element.
        reverse: If True, sort in descending order.

    Returns:
        A new list containing the sorted elements.
    """
    if len(arr) <= 1:
        return list(arr)

    key_func = key or _identity

    mid = len(arr) // 2
    left = merge_sort(arr[:mid], key=key_func, reverse=reverse)
    right = merge_sort(arr[mid:], key=key_func, reverse=reverse)

    left_with_keys = [(v, key_func(v)) for v in left]
    right_with_keys = [(v, key_func(v)) for v in right]

    merged = []
    i = j = 0

    # Maintain stable order: <= for ascending, >= for descending
    while i < len(left_with_keys) and j < len(right_with_keys):
        lv, lk = left_with_keys[i]
        rv, rk = right_with_keys[j]

        if (lk <= rk and not reverse) or (lk >= rk and reverse):
            merged.append(lv)
            i += 1
        else:
            merged.append(rv)
            j += 1

    merged.extend(v for v, _ in left_with_keys[i:])
    merged.extend(v for v, _ in right_with_keys[j:])
    return merged


# Minimal tests for automated checking
if __name__ == "__main__":
    sample_int = [5, 2, 9, 1, 5, 6]
    sample_str = ['apple', 'fig', 'banana']
    empty_list = []
    duplicate_keys = ['a', 'A', 'b', 'B', 'a']

    assert merge_sort(sample_int) == sorted(sample_int)
    assert merge_sort(sample_int, reverse=True) == sorted(sample_int, reverse=True)
    assert merge_sort(sample_str, key=len) == sorted(sample_str, key=len)
    assert merge_sort(empty_list) == []
    assert merge_sort(duplicate_keys, key=str.lower) == sorted(duplicate_keys, key=str.lower)

    print("All tests passed.")
```

In [ ]:
#| echo: false
display_markdown(output["reflection_history"][2]["content"], raw=True)